In [ ]:
!pip install trl
!pip install -q wandb

import gc
import shutil
import wandb
import sys
import os
import re
import time
import json
import math
import random
import numpy as np
import pandas as pd
import torch

from copy import deepcopy
from pprint import pprint
from dataclasses import dataclass
from typing import List, Optional, Dict, Any, Union
from tqdm.auto import tqdm
from datasets import load_dataset, Dataset, DatasetDict
from huggingface_hub import list_repo_files, hf_hub_download
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
)
from trl import SFTTrainer, SFTConfig
from sklearn.model_selection import train_test_split

In [ ]:
os.environ["WANDB_API_KEY"] = "Your wandb key"
wandb.login()

In [ ]:
PROJECT_DIR = "/kaggle/input/datasets/skimaks/project/sft_refactor"
os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)

In [ ]:
from sft.config import DataConfig, ModelConfig, TrainConfig
from sft.pipeline import (
    prepare_data_pipeline,
    prepare_tokenizer_pipeline,
    render_dataset_with_chat_template,
)
from sft.model_utils import load_model_for_sft, sync_model_embeddings_with_tokenizer
from sft.training import build_training_args, build_trainer, train_once

from sft.metrics import json_validity_rate, function_call_em, debug_json_generation, generate_text
from sft.config import ModelConfig, DataConfig


# ЗАДАНИЕ 1 - Два варианта токенизации текста

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
# Загружаем токенизатор модели
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
# Если PAD токен не задан, используем EOS токен в качестве PAD
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

In [ ]:
def naive_tokenize(text: str, tokenizer) -> List[int]:
    """
    ЗАДАНИЕ: Реализуйте наивную токенизацию, где каждое слово (разделённое пробелами)
    ищется целиком в словаре токенизатора (vocab).

    Этот подход демонстрирует НЕПРАВИЛЬНЫЙ способ токенизации, чтобы показать
    разницу с правильным методом (encode/decode).

    Алгоритм:
    1. Разбейте текст по пробелам с помощью функции split()
    2. Для каждого слова проверьте, есть ли оно в vocab токенизатора
    3. Если слово найдено - добавьте его ID в список
    4. Если слово не найдено - добавьте -1 (маркер "неизвестного" токена)

    Проблемы этого подхода:
    - BPE/WordPiece работают по подсловам, а не целым словам
    - В vocab нет целых слов с пунктуацией/регистром
    - Игнорируются спецтокены и нормализация

    Args:
        text: входной текст для токенизации
        tokenizer: токенизатор модели

    Returns:
        List[int]: список ID токенов (или -1 для неизвестных слов)
    """
    vocab = tokenizer.get_vocab()
    tokens = text.split()

    result = []
    for word in tokens:
        if word in vocab:
            result.append(vocab[word])
        else:
            result.append(-1)

    return result


try:
    en = "Do or do not, there is no try"
    ru = "Делай или не делай — попыток нет"
    print("EN (наивно):", naive_tokenize(en, tokenizer))
    print("RU (наивно):", naive_tokenize(ru, tokenizer))
except Exception as e:
    print("Ожидаемая ошибка из‑за неправильного подхода:", e)

#### Вывод: наивная токенизация работает плохо, особенно для русского текста. Для английского она смогла найти в словаре только часть слов, а одно слово оказалось неизвестным (-1). Для русского все слова оказались неизвестными, потому что токенизатор такой модели обычно использует подсловную токенизацию (BPE/WordPiece), а не хранит целые слова в словаре. Это показывает, что искать слова целиком по vocab — неверный подход

In [ ]:
def inspect_tokenization(text: str, tokenizer) -> None:
    """
    ЗАДАНИЕ: Реализуйте правильную токенизацию с использованием методов encode/decode.

    Этот подход демонстрирует ПРАВИЛЬНЫЙ способ токенизации, который использует
    встроенные методы токенизатора для корректной обработки текста.

    Алгоритм:
    1. Используйте метод encode() для преобразования текста в ID токенов
       (без добавления специальных токенов - за это отвечает отдельный параметр в encode)
    2. Используйте метод convert_ids_to_tokens() для получения строковых представлений токенов
    3. Используйте метод decode() для обратного преобразования ID в текст
       (пропуская специальные токены - за это отвечает отдельный параметр в decode)
    4. Выведите исходный текст, ID токенов, строковые токены и декодированный текст
    5. Проверьте, совпадает ли декодированный текст с исходным

    Преимущества этого подхода:
    - Корректная работа с BPE/WordPiece подсловами
    - Правильная обработка пунктуации и регистра
    - Учёт нормализации и специальных токенов
    - Обратимость: encode → decode возвращает исходный текст

    Args:
        text: входной текст для токенизации
        tokenizer: токенизатор модели

    Returns:
        None (функция выводит результаты через print)
    """
    token_ids = tokenizer.encode(text, add_special_tokens=False)
    tokens = tokenizer.convert_ids_to_tokens(token_ids)
    decoded_text = tokenizer.decode(token_ids, skip_special_tokens=True)

    print("Исходный текст:", text)
    print("ID токенов:", token_ids)
    print("Строковые токены:", tokens)
    print("Декодированный текст:", decoded_text)
    print("Совпадает ли decode с исходным:", decoded_text == text)


# Демонстрация (без строгой проверки регистров/нормализации):
en = "Do or do not, there is no try"
ru = "Делай или не делай — попыток нет"
print("\nEN:")
inspect_tokenization(en, tokenizer)
print("\nRU:")
inspect_tokenization(ru, tokenizer)

#### Вывод: правильная токенизация через encode/decode работает корректно и для английского, и для русского текста. В обоих случаях текст разбивается не на целые слова, а на подсловные токены, что особенно хорошо видно на русском примере. Несмотря на то что строковые токены выглядят непривычно и содержат технические символы, декодирование полностью восстанавливает исходную строку. Это показывает главное преимущество правильного подхода: токенизатор умеет корректно работать с BPE-представлением текста и сохраняет обратимость encode → decode.

# Задание 2 - Обучение и оценка модели

In [ ]:
# создание конфига данных и подготовка датасета
data_cfg = DataConfig()
prepared_data = prepare_data_pipeline(data_cfg)

print(type(prepared_data))
print(prepared_data.dataset_messages)

In [ ]:
# подготовка токенизатора и превращение messages в текст через chat template.
model_cfg = ModelConfig()
prepared_tokenizer = prepare_tokenizer_pipeline(model_cfg)

dataset_text = render_dataset_with_chat_template(
    prepared_data.dataset_messages,
    prepared_tokenizer.tokenizer,
)

print(dataset_text)
print(dataset_text["train"][0])

In [ ]:
# ===== JSON validity prompts из исходного ноутбука =====
json_prompt_template = """Верни ТОЛЬКО чистый JSON-объект без Markdown, кода, текста или объяснений.
Ключи: "name" (строка, название темы), "count" (целое число, примерное количество).
По теме: {topic}. Пример: {{"name": "{topic}", "count": 5}}"""

json_topics = [
    "машинное обучение",
    "компьютерное зрение",
    "обработка текста",
    "математика",
    "космос",
    "история",
    "кулинария",
    "спорт",
    "музыка",
    "кино",
]

json_prompts = [json_prompt_template.format(topic=t) for t in (json_topics * 10)]

print("JSON prompts:", len(json_prompts))


# ===== Function Calling EM@20 cases из исходного ноутбука =====
fc_cases = []

cities = ["Москва", "Казань", "Томск", "Сочи", "Пермь"]
for c in cities:
    fc_cases.append(
        {
            "prompt": f'Верни ТОЛЬКО чистый JSON без Markdown и текста. Функция get_weather для города {c}. Формат: {{"name": "get_weather", "arguments": {{"city": "{c}"}}}}',
            "expected": {"name": "get_weather", "arguments": {"city": c}},
        }
    )

fc_cases += [
    {
        "prompt": 'Верни ТОЛЬКО чистый JSON без Markdown и текста. Функция sum с аргументами a=2, b=7. Формат: {"name": "sum", "arguments": {"a": 2, "b": 7}}',
        "expected": {"name": "sum", "arguments": {"a": 2, "b": 7}},
    },
    {
        "prompt": 'Верни ТОЛЬКО чистый JSON без Markdown и текста. Функция sum с аргументами a=3, b=5. Формат: {"name": "sum", "arguments": {"a": 3, "b": 5}}',
        "expected": {"name": "sum", "arguments": {"a": 3, "b": 5}},
    },
    {
        "prompt": 'Верни ТОЛЬКО чистый JSON без Markdown и текста. Функция search с query=\'перплексия\'. Формат: {"name": "search", "arguments": {"query": "перплексия"}}',
        "expected": {"name": "search", "arguments": {"query": "перплексия"}},
    },
    {
        "prompt": 'Верни ТОЛЬКО чистый JSON без Markdown и текста. Функция search с query=\'LoRA\'. Формат: {"name": "search", "arguments": {"query": "LoRA"}}',
        "expected": {"name": "search", "arguments": {"query": "LoRA"}},
    },
]

while len(fc_cases) < 20:
    fc_cases.append(
        {
            "prompt": 'Верни ТОЛЬКО чистый JSON без Markdown и текста. Функция sum с аргументами a=1, b=1. Формат: {"name": "sum", "arguments": {"a": 1, "b": 1}}',
            "expected": {"name": "sum", "arguments": {"a": 1, "b": 1}},
        }
    )

print("FC cases:", len(fc_cases))
print(json_prompts[0])
print(fc_cases[0])

In [ ]:
def cleanup_after_run(output_dir=None, model=None, trainer=None):
    if trainer is not None:
        del trainer
    if model is not None:
        del model

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    if output_dir is not None and os.path.exists(output_dir):
        shutil.rmtree(output_dir, ignore_errors=True)

In [ ]:
def resolve_torch_dtype(torch_dtype: str):
    mapping = {
        "fp16": torch.float16,
        "float16": torch.float16,
        "bf16": torch.bfloat16,
        "bfloat16": torch.bfloat16,
        "fp32": torch.float32,
        "float32": torch.float32,
    }
    if torch_dtype not in mapping:
        raise ValueError(f"Unsupported torch_dtype={torch_dtype}")
    return mapping[torch_dtype]


In [ ]:
def make_wandb_config(
    data_cfg: DataConfig,
    model_cfg: ModelConfig,
    train_cfg: TrainConfig,
    packing: bool,
    torch_dtype: str,
    run_name: str,
):
    return {
        "run_name": run_name,
        "dataset_name": data_cfg.dataset_name,
        "num_samples": data_cfg.num_samples,
        "val_ratio": data_cfg.val_ratio,
        "seed": data_cfg.seed,
        "tool_injection_prob": data_cfg.tool_injection_prob,
        "multi_turn_dialogues": data_cfg.multi_turn_dialogues,
        "model_name": model_cfg.model_name,
        "trust_remote_code": model_cfg.trust_remote_code,
        "resize_vocab_to_multiple_of": model_cfg.resize_vocab_to_multiple_of,
        "torch_dtype": torch_dtype,
        "num_epochs": train_cfg.num_epochs,
        "learning_rate": train_cfg.learning_rate,
        "batch_size": train_cfg.batch_size,
        "grad_accum_steps": train_cfg.grad_accum_steps,
        "max_seq_length": train_cfg.max_seq_length,
        "warmup_ratio": train_cfg.warmup_ratio,
        "max_grad_norm": train_cfg.max_grad_norm,
        "logging_steps": train_cfg.logging_steps,
        "eval_steps": train_cfg.eval_steps,
        "save_steps": train_cfg.save_steps,
        "lr_scheduler_type": train_cfg.lr_scheduler_type,
        "report_to": train_cfg.report_to,
        "packing": packing,
    }

In [ ]:
def load_fresh_model_for_run(
    model_cfg: ModelConfig,
    tokenizer,
    torch_dtype: str,
):
    dtype = resolve_torch_dtype(torch_dtype)

    model = AutoModelForCausalLM.from_pretrained(
        model_cfg.model_name,
        trust_remote_code=model_cfg.trust_remote_code,
        torch_dtype=dtype,
    )
    model.gradient_checkpointing_enable()
    model.config.use_cache = False

    resized = sync_model_embeddings_with_tokenizer(model, tokenizer)
    return model, resized

In [ ]:
def extract_training_summary(train_result):
    train_metrics = train_result.train_result.metrics
    eval_metrics = train_result.eval_metrics

    return {
        "final_train_loss": train_result.train_result.training_loss,
        "final_eval_loss": train_result.eval_loss,
        "perplexity": train_result.perplexity,
        "train_runtime": train_metrics.get("train_runtime"),
        "train_samples_per_second": train_metrics.get("train_samples_per_second"),
        "train_steps_per_second": train_metrics.get("train_steps_per_second"),
        "eval_runtime": eval_metrics.get("eval_runtime"),
        "eval_samples_per_second": eval_metrics.get("eval_samples_per_second"),
        "eval_steps_per_second": eval_metrics.get("eval_steps_per_second"),
    }

In [ ]:
def compute_custom_metrics(
    model,
    tokenizer,
    json_prompts,
    fc_cases,
):
    json_valid = json_validity_rate(
        model=model,
        tokenizer=tokenizer,
        prompts=json_prompts,
        n=100,
        robust=True,
    )

    em20 = function_call_em(
        model=model,
        tokenizer=tokenizer,
        cases=fc_cases,
        verbose=False,
    )

    return {
        "json_validity_100": json_valid,
        "function_call_em_20": em20,
    }

In [ ]:
def prepare_pipeline_once(
    data_cfg: DataConfig,
    model_cfg: ModelConfig,
):
    prepared_data = prepare_data_pipeline(data_cfg)
    prepared_tokenizer = prepare_tokenizer_pipeline(model_cfg)
    dataset_text = render_dataset_with_chat_template(
        prepared_data.dataset_messages,
        prepared_tokenizer.tokenizer,
    )
    return prepared_data, prepared_tokenizer, dataset_text

In [ ]:
def run_sft_experiment(
    *,
    run_name: str,
    project_name: str,
    json_prompts,
    fc_cases,
    packing: bool = False,
    torch_dtype: str = "bf16",
    output_root: str = "/kaggle/working/sft_output",
    tags=None,
    data_cfg: DataConfig | None = None,
    model_cfg: ModelConfig | None = None,
    train_cfg: TrainConfig | None = None,
    prepared_data=None,
    prepared_tokenizer=None,
    dataset_text=None,
):
    if data_cfg is None:
        data_cfg = DataConfig()
    else:
        data_cfg = deepcopy(data_cfg)

    if model_cfg is None:
        model_cfg = ModelConfig()
    else:
        model_cfg = deepcopy(model_cfg)

    if train_cfg is None:
        train_cfg = TrainConfig()
    else:
        train_cfg = deepcopy(train_cfg)

    train_cfg.output_root = output_root
    train_cfg.report_to = "wandb"

    if prepared_data is None or prepared_tokenizer is None or dataset_text is None:
        prepared_data, prepared_tokenizer, dataset_text = prepare_pipeline_once(
            data_cfg=data_cfg,
            model_cfg=model_cfg,
        )

    run = wandb.init(
        project=project_name,
        name=run_name,
        config=make_wandb_config(
            data_cfg=data_cfg,
            model_cfg=model_cfg,
            train_cfg=train_cfg,
            packing=packing,
            torch_dtype=torch_dtype,
            run_name=run_name,
        ),
        tags=tags or [],
    )

    model = None
    trainer = None
    output_dir = os.path.join(output_root, run_name)

    try:
        model, resized = load_fresh_model_for_run(
            model_cfg=model_cfg,
            tokenizer=prepared_tokenizer.tokenizer,
            torch_dtype=torch_dtype,
        )

        run.config.update(
            {
                "model_dtype_runtime": str(model.dtype),
                "resized_embeddings_by": resized,
                "train_rows": len(dataset_text["train"]),
                "val_rows": len(dataset_text["validation"]),
                "tool_stats": prepared_data.tool_stats,
                "augmentation_report": prepared_data.augmentation_report,
            },
            allow_val_change=True,
        )

        train_args = build_training_args(
            train_cfg=train_cfg,
            output_dir=output_dir,
            packing=packing,
        )

        trainer = build_trainer(
            model=model,
            tokenizer=prepared_tokenizer.tokenizer,
            train_args=train_args,
            dataset=dataset_text,
        )

        train_result = train_once(trainer)

        summary = {
            "run_name": run_name,
            "packing": packing,
            "torch_dtype": torch_dtype,
            **extract_training_summary(train_result),
            **compute_custom_metrics(
                model=model,
                tokenizer=prepared_tokenizer.tokenizer,
                json_prompts=json_prompts,
                fc_cases=fc_cases,
            ),
        }

        wandb.log(summary)
        for k, v in summary.items():
            run.summary[k] = v

        return {
            "summary": summary,
        }

    finally:
        wandb.finish()
        cleanup_after_run(output_dir=output_dir, model=model, trainer=trainer)

In [ ]:
data_cfg = DataConfig()
model_cfg = ModelConfig()
train_cfg = TrainConfig()

data_cfg.num_samples = 1000
train_cfg.num_epochs = 1
train_cfg.learning_rate = 2e-5
train_cfg.batch_size = 1
train_cfg.grad_accum_steps = 4
train_cfg.eval_steps = 20
train_cfg.logging_steps = 10

prepared_data, prepared_tokenizer, dataset_text = prepare_pipeline_once(
    data_cfg=data_cfg,
    model_cfg=model_cfg,
)

In [ ]:
out = run_sft_experiment(
    run_name="my_new_run",
    project_name="sft-homework",
    json_prompts=json_prompts,
    fc_cases=fc_cases,
    packing=False,
    torch_dtype="bf16",
    tags=["test"],
    data_cfg=data_cfg,
    model_cfg=model_cfg,
    train_cfg=train_cfg,
    prepared_data=prepared_data,
    prepared_tokenizer=prepared_tokenizer,
    dataset_text=dataset_text,
)

out["summary"]

In [ ]:
ENTITY = "maximdegtyarev1996-avito"
PROJECT = "sft-homework"

In [ ]:
api = wandb.Api()

runs = api.runs(f"{ENTITY}/{PROJECT}")

rows = []
for run in runs:
    cfg = dict(run.config)
    summ = dict(run.summary)

    rows.append({
        "run_id": run.id,
        "run_name": run.name,
        "state": run.state,
        "url": run.url,

        # конфиг
        "model_name": cfg.get("model_name"),
        "num_epochs": cfg.get("num_epochs"),
        "learning_rate": cfg.get("learning_rate"),
        "batch_size": cfg.get("batch_size"),
        "grad_accum_steps": cfg.get("grad_accum_steps"),
        "packing": cfg.get("packing"),
        "torch_dtype": cfg.get("torch_dtype"),
        "dataset_name": cfg.get("dataset_name"),
        "max_seq_length": cfg.get("max_seq_length"),

        # итоговые метрики
        "final_train_loss": summ.get("final_train_loss"),
        "final_eval_loss": summ.get("final_eval_loss"),
        "perplexity": summ.get("perplexity"),
        "json_validity_100": summ.get("json_validity_100"),
        "function_call_em_20": summ.get("function_call_em_20"),
        "train_runtime": summ.get("train_runtime"),
        "train_samples_per_second": summ.get("train_samples_per_second"),
        "train_steps_per_second": summ.get("train_steps_per_second"),
        "eval_runtime": summ.get("eval_runtime"),
        "eval_samples_per_second": summ.get("eval_samples_per_second"),
        "eval_steps_per_second": summ.get("eval_steps_per_second"),
    })

runs_df = pd.DataFrame(rows)
runs_df

In [ ]:
config_df = runs_df[
    [
        "run_name",
        "model_name",
        "num_epochs",
        "learning_rate",
        "batch_size",
        "grad_accum_steps",
        "packing",
        "torch_dtype",
        "dataset_name",
        "max_seq_length",
    ]
].copy()

metrics_df = runs_df[
    [
        "run_name",
        "train_runtime",
        "train_samples_per_second",
        "train_steps_per_second",
        "final_train_loss",
        "final_eval_loss",
        "perplexity",
        "json_validity_100",
        "function_call_em_20",
    ]
].copy()

In [ ]:
config_df.to_csv("/kaggle/working/wandb_experiment_configs.csv", index=False)
metrics_df.to_csv("/kaggle/working/wandb_experiment_metrics.csv", index=False)

print("saved:",
      "/kaggle/working/wandb_experiment_configs.csv",
      "/kaggle/working/wandb_experiment_metrics.csv")

### Краткий вывод

Лучшим запуском оказался **`grid6_run3_ep1_lr2e-05_bs1_bf16_packFalse`**, показавший минимальный `eval loss = 1.6236` и `perplexity = 5.0715`.

В рамках рассмотренной сетки качество улучшалось при увеличении learning rate до `2e-5`.  
Режим с `batch_size = 1` consistently показывал более низкий `eval loss`, чем `batch_size = 2`, хотя `batch_size = 2` обеспечивал более высокую скорость обучения.

Использование `packing=True` привело к заметному ускорению обучения, но резко ухудшило качество модели, поэтому в данном эксперименте этот режим оказался неэффективным.

In [ ]:
# подготовка конфига лучшего запуска
best_data_cfg = deepcopy(data_cfg)
best_model_cfg = deepcopy(model_cfg)
best_train_cfg = deepcopy(train_cfg)

best_data_cfg.num_samples = 1000

best_train_cfg.num_epochs = 1
best_train_cfg.learning_rate = 2e-5
best_train_cfg.batch_size = 1
best_train_cfg.grad_accum_steps = 4
best_train_cfg.eval_steps = 20
best_train_cfg.logging_steps = 10
best_train_cfg.output_root = "/kaggle/working/sft_output_best"

BEST_RUN_NAME = "grid6_run3_ep1_lr2e-05_bs1_bf16_packFalse"
BEST_OUTPUT_DIR = "/kaggle/working/sft_output_best/final_model"

In [ ]:
# подготовка данных и токенизатора
prepared_data_best, prepared_tokenizer_best, dataset_text_best = prepare_pipeline_once(
    data_cfg=best_data_cfg,
    model_cfg=best_model_cfg,
)

print(dataset_text_best)

In [ ]:
# загрузка модели
model_best, resized_best = load_fresh_model_for_run(
    model_cfg=best_model_cfg,
    tokenizer=prepared_tokenizer_best.tokenizer,
    torch_dtype="bf16",
)

print("resized_best =", resized_best)
print("model dtype =", model_best.dtype)

In [ ]:
# собираем trainer и обучаем лучшую модель
best_args = build_training_args(
    train_cfg=best_train_cfg,
    output_dir="/kaggle/working/sft_output_best/train_artifacts",
    packing=False,
)

best_trainer = build_trainer(
    model=model_best,
    tokenizer=prepared_tokenizer_best.tokenizer,
    train_args=best_args,
    dataset=dataset_text_best,
)

best_result = train_once(best_trainer)

print(best_result)

In [ ]:
# cохранение модели и токенизатора
os.makedirs(BEST_OUTPUT_DIR, exist_ok=True)

best_trainer.save_model(BEST_OUTPUT_DIR)
prepared_tokenizer_best.tokenizer.save_pretrained(BEST_OUTPUT_DIR)

print("saved to:", BEST_OUTPUT_DIR)

In [ ]:
# примеры генераций после SFT

test_prompts = [
    "Объясни, что такое машинное обучение простыми словами.",
    "Напиши короткое стихотворение о зиме.",
    "Как приготовить омлет?",
    "В чем разница между Python и JavaScript?",
    "Расскажи интересный факт о космосе.",
]

examples = []

for prompt in test_prompts:
    response = generate_text(
        model=model_best,
        tokenizer=prepared_tokenizer_best.tokenizer,
        prompt=prompt,
        max_new_tokens=256,
        do_sample=True,
    )
    examples.append({
        "prompt": prompt,
        "response": response,
    })

examples[:3]

In [ ]:
# сохранение примеров генераций после STF
GEN_PATH = "/kaggle/working/sft_output_best/generation_examples.json"

with open(GEN_PATH, "w", encoding="utf-8") as f:
    json.dump(examples, f, ensure_ascii=False, indent=2)

print("saved:", GEN_PATH)

In [ ]:
# генерации модели до sft

BEFORE_ROOT = "/kaggle/working/sft_output_before"
os.makedirs(BEFORE_ROOT, exist_ok=True)

base_model = AutoModelForCausalLM.from_pretrained(
    model_cfg.model_name,
    trust_remote_code=model_cfg.trust_remote_code,
    torch_dtype=torch.bfloat16,
)

base_model.gradient_checkpointing_enable()
base_model.config.use_cache = False

# токенизатор уже расширен спецтокенами, поэтому синхронизируем эмбеддинги
_ = sync_model_embeddings_with_tokenizer(base_model, prepared_tokenizer.tokenizer)

before_sft_examples = []

for i, prompt in enumerate(test_prompts, start=1):
    print(f"[{i}/{len(test_prompts)}] {prompt}")
    response = generate_text(
        model=base_model,
        tokenizer=prepared_tokenizer.tokenizer,
        prompt=prompt,
        max_new_tokens=256,
        do_sample=True,
    )
    before_sft_examples.append({
        "prompt": prompt,
        "before_sft": response,
    })

BEFORE_JSON_PATH = f"{BEFORE_ROOT}/generation_examples_before_sft.json"

with open(BEFORE_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(before_sft_examples, f, ensure_ascii=False, indent=2)

print("Saved to:", BEFORE_JSON_PATH)

print("\nPreview:")
for ex in before_sft_examples:
    print("\nPROMPT:", ex["prompt"])
    print("BEFORE:", ex["before_sft"][:200])